# Week 10: Domain Features (m5)

Follow-up to `wk9_pipeline_overhaul.ipynb`. That notebook audited the pipeline mechanics
(leakage, encoding, hyperparameters). This one asks a different question: are there
housing-domain-specific features the model is missing entirely, independent of any bug?

Prompted by `AVM_Data_Science_Best_Practices_v.1.pdf` (Sections 05-06), which names three
feature families -- intrinsic, locational, temporal -- and this pipeline never had a real
temporal feature at all: `SaleYearMonth` was used only to build the chronological split,
never fed to any model as an input. That's the headline gap this notebook closes.

**New features added, each with before/after evidence:**
1. Cyclical month encoding (`sin`/`cos`) plus a continuous time-trend feature -- the model's
   first exposure to seasonality and market drift.
2. A ZIP/area price-per-sqft "comps" feature, computed on training data only and joined
   forward (Section 05, Locational Features) -- closer to how a human appraiser actually
   prices a home than an opaque target-encoded category mean.
3. Distance to the nearest major California employment center, computed from lat/long
   (Section 05) -- a fixed, pre-sale-known geometric fact, not a data-derived statistic, so
   it carries zero leakage risk regardless of which rows land in train vs. test.
4. Missing-indicator flags for `AssociationFee` and `GarageSpaces` (Section 06) -- missingness
   itself may be informative and was previously being thrown away by zero-fill imputation.

Also adds a rolling-origin backtest (Section 01) on the winning model, something the wk9
notebook never did: every number reported there came from a single train/val/test cutoff,
and a model that looks great on one cutoff can be unstable on the next.

**Reproducibility note.** This uses the same data snapshot as the corrected wk9 notebook --
29 monthly files through `CRMLSSold202605.csv` (May 2026) -- so m5 results below are directly
comparable to wk9's m4 numbers (LightGBM R2=0.9053, MAPE=11.78%), not the earlier sandbox
run that used a different snapshot. `N_TRAIN_MONTHS` is reused at 24 (wk9's finding for this
snapshot) rather than re-swept here -- worth re-checking in a future pass since the new
features could plausibly shift the optimal window, but re-sweeping wasn't repeated to keep
this notebook focused on the features themselves.

## 1. Setup and Imports

In [1]:
import os
import re
import glob
import time
import math
import numpy as np
import pandas as pd
import geopandas as gpd
from word2number import w2n
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42

os.chdir(os.path.expanduser("~/Desktop/CAPropPredictor"))

## 2-9. Ingestion Through School-District Join  ·  Unchanged From wk9

Sections 2-9 are the identical, already-verified wk9 pipeline (raw ingestion, exclusions,
null-rate filter, row-quality checks including the bed/bath cap, lot-size check, type
parsing, geocoding + null-island filter, and the school-district join fix). Reproduced
here rather than imported so this notebook stays runnable start-to-finish on its own, matching
the style of every other notebook in this project.

One deliberate change from wk9: `PostalCode` is kept (not dropped in the "redundant location"
pass) because it's needed for the new ZIP-level comps feature in Section 11.

In [2]:
file_paths = sorted(glob.glob("CRMLSData/*.csv"))
dataframes = [pd.read_csv(f, low_memory=False) for f in file_paths]
merged_df = pd.concat(dataframes, ignore_index=True)
print(f"merged raw: {merged_df.shape} from {len(file_paths)} files")

housing_scoped = merged_df[
    (merged_df["PropertyType"] == "Residential")
    & (merged_df["PropertySubType"] == "SingleFamilyResidence")
].copy()
housing_scoped["SaleYearMonth"] = pd.to_datetime(housing_scoped["CloseDate"]).dt.to_period("M")
print(f"scoped: {housing_scoped.shape}")

merged raw: (636443, 82) from 29 files
scoped: (320506, 83)


In [3]:
agent_and_office_identity_columns = [
    "ListAgentEmail", "ListAgentFullName", "ListAgentFirstName", "ListAgentLastName",
    "ListAgentAOR", "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerAgentFirstName", "BuyerAgentLastName", "BuyerAgentMlsId", "BuyerAgentAOR",
    "CoBuyerAgentFirstName", "ListOfficeName", "BuyerOfficeName", "BuyerOfficeAOR",
]
business_scope_columns = ["BusinessType"]
regional_architecture_columns = ["AboveGradeFinishedArea", "BelowGradeFinishedArea"]
post_filter_scope_columns = ["PropertyType", "PropertySubType"]
leakage_columns = [
    "ListPrice", "OriginalListPrice", "DaysOnMarket",
    "CloseDate", "ContractStatusChangeDate", "PurchaseContractDate", "ListingContractDate",
]
redundant_location_columns = ["UnparsedAddress"]
uninformative_columns = ["MlsStatus", "latfilled", "lonfilled"]
unreliable_columns = ["MainLevelBedrooms"]

excluded_feature_columns = (
    agent_and_office_identity_columns + business_scope_columns + regional_architecture_columns
    + post_filter_scope_columns + leakage_columns + redundant_location_columns
    + uninformative_columns + unreliable_columns
)
cols_before = housing_scoped.shape[1]
housing_after_exclusions = housing_scoped.drop(columns=[c for c in excluded_feature_columns if c in housing_scoped.columns])
print(f"dropped {len(excluded_feature_columns)} columns ({cols_before} -> {housing_after_exclusions.shape[1]})")

dropped 32 columns (83 -> 51)


In [4]:
NULL_RATE_THRESHOLD = 0.60
null_rate_by_column = housing_after_exclusions.isnull().sum() / len(housing_after_exclusions)
high_null_rate_columns = null_rate_by_column[null_rate_by_column > NULL_RATE_THRESHOLD].index.tolist()
housing_after_null_filter = housing_after_exclusions.drop(columns=high_null_rate_columns)
print(f"dropped {len(high_null_rate_columns)} columns for null rate > {NULL_RATE_THRESHOLD:.0%} -> {housing_after_null_filter.shape[1]} cols")

dropped 19 columns for null rate > 60% -> 32 cols


In [5]:
housing_step = housing_after_null_filter
before = len(housing_step)
duplicate_count = housing_step.duplicated(subset=["ListingKey"]).sum()
housing_step = housing_step.drop_duplicates(subset=["ListingKey"])
print(f"duplicates: dropped {duplicate_count} ({before} -> {len(housing_step)})")

before = len(housing_step)
invalid_target_mask = housing_step["ClosePrice"].isna() | (housing_step["ClosePrice"] <= 0)
housing_step = housing_step[~invalid_target_mask]
print(f"invalid ClosePrice: dropped {invalid_target_mask.sum()} ({before} -> {len(housing_step)})")

before = len(housing_step)
zero_or_negative_sqft_mask = housing_step["LivingArea"] <= 0
SQFT_PER_BEDROOM_FLOOR = 70
implausible_bedroom_density_mask = (
    (housing_step["BedroomsTotal"] > 0)
    & (housing_step["LivingArea"] / housing_step["BedroomsTotal"] < SQFT_PER_BEDROOM_FLOOR)
)
negative_bathroom_mask = housing_step["BathroomsTotalInteger"] < 0
BATHROOM_CAP, BEDROOM_CAP = 10, 10
implausible_bathroom_count_mask = housing_step["BathroomsTotalInteger"] > BATHROOM_CAP
implausible_bedroom_count_mask = housing_step["BedroomsTotal"] > BEDROOM_CAP

logical_impossibility_mask = (
    zero_or_negative_sqft_mask | implausible_bedroom_density_mask | negative_bathroom_mask
    | implausible_bathroom_count_mask | implausible_bedroom_count_mask
)
housing_step = housing_step[~logical_impossibility_mask]
print(f"logical impossibilities: dropped {logical_impossibility_mask.sum()} ({before} -> {len(housing_step)})")

before = len(housing_step)
non_ca_mask = housing_step["StateOrProvince"] != "CA"
housing_step = housing_step[~non_ca_mask]
print(f"non-CA: dropped {non_ca_mask.sum()} ({before} -> {len(housing_step)})")
housing_after_state_filter = housing_step

duplicates: dropped 276 (320506 -> 320230)
invalid ClosePrice: dropped 3 (320230 -> 320227)
logical impossibilities: dropped 335 (320227 -> 319892)
non-CA: dropped 22 (319892 -> 319870)


In [6]:
ACRE_TO_SQFT = 43560
sqft_null_mask = housing_after_state_filter["LotSizeSquareFeet"].isna()
lot_size_drop_columns = [c for c in ["LotSizeAcres", "LotSizeArea"] if c in housing_after_state_filter.columns]
housing_after_lot_size_reconciliation = housing_after_state_filter.drop(columns=lot_size_drop_columns)

# NOTE: PostalCode is deliberately KEPT here (unlike wk9) -- needed for the Section 11 comps feature
second_pass_drop_columns = [c for c in ["StreetNumberNumeric", "StateOrProvince", "ListingKeyNumeric", "ListingId"] if c in housing_after_lot_size_reconciliation.columns]
housing_after_second_pass_drop = housing_after_lot_size_reconciliation.drop(columns=second_pass_drop_columns)
print(f"-> {housing_after_second_pass_drop.shape[1]} cols (PostalCode kept for Section 11 comps feature)")

-> 26 cols (PostalCode kept for Section 11 comps feature)


In [7]:
def general_numeric_parser(val):
    if pd.isna(val) or val == '':
        return 0
    val_str = str(val).strip()
    parts = [p.strip() for p in val_str.split(',')]
    found_numbers = []
    for part in parts:
        part_clean = part.lower()
        try:
            clean_word = part_clean.replace("ormore", "").replace("plus", "").strip()
            num = w2n.word_to_num(clean_word)
            found_numbers.append(num)
            continue
        except ValueError:
            pass
        digits = re.findall(r'\d+', part_clean)
        if digits:
            found_numbers.append(int(digits[0]))
            continue
    if found_numbers:
        return max(found_numbers)
    return 0

def general_boolean_parser(val):
    if pd.isna(val):
        return 0
    val_clean = str(val).strip().lower()
    truth_values = {'true', 't', 'yes', 'y', '1', '1.0'}
    false_values = {'false', 'f', 'no', 'n', '0', '0.0'}
    if val_clean in truth_values:
        return 1
    if val_clean in false_values:
        return 0
    return 0

def is_safe_to_cast_int(series):
    non_null = series.dropna()
    if non_null.empty:
        return True
    return (non_null % 1 == 0).all()

intrinsic_bool_cols = [c for c in ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN"] if c in housing_after_second_pass_drop.columns]
intrinsic_word_encoded_cols = [c for c in ["Levels"] if c in housing_after_second_pass_drop.columns]
intrinsic_count_cols = [c for c in ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces", "ParkingTotal", "Stories"] if c in housing_after_second_pass_drop.columns]

housing_after_type_parsing = housing_after_second_pass_drop.copy()
for col in intrinsic_bool_cols:
    housing_after_type_parsing[col] = housing_after_type_parsing[col].apply(general_boolean_parser)
for col in intrinsic_word_encoded_cols:
    housing_after_type_parsing[col] = housing_after_type_parsing[col].apply(general_numeric_parser)
for col in intrinsic_count_cols:
    if is_safe_to_cast_int(housing_after_type_parsing[col]):
        housing_after_type_parsing[col] = housing_after_type_parsing[col].astype("Int64")

print(f"type parsing done: {housing_after_type_parsing.shape}")

type parsing done: (319870, 26)


In [8]:
def find_rows_needing_geocoding(df, min_cluster_size=5):
    missing_mask = df["Latitude"].isna() | df["Longitude"].isna()
    coord_counts = df.groupby(["Latitude", "Longitude"])["ListingKey"].transform("size")
    suspect_cluster_mask = (coord_counts >= min_cluster_size) & ~missing_mask
    return missing_mask | suspect_cluster_mask

needs_geocoding = find_rows_needing_geocoding(housing_after_type_parsing)
geocode_checkpoint = pd.read_csv("CRMLSCleaned/m3_geocode_checkpoint.csv")
housing_after_type_parsing = housing_after_type_parsing.merge(
    geocode_checkpoint[["ListingKey", "geocoded_lat", "geocoded_lon", "coord_status"]],
    on="ListingKey", how="left",
)
confirmed_mask = housing_after_type_parsing["coord_status"] == "filled_from_geocode"
housing_after_type_parsing.loc[confirmed_mask, "Latitude"] = housing_after_type_parsing.loc[confirmed_mask, "geocoded_lat"]
housing_after_type_parsing.loc[confirmed_mask, "Longitude"] = housing_after_type_parsing.loc[confirmed_mask, "geocoded_lon"]
housing_after_type_parsing = housing_after_type_parsing.drop(columns=["geocoded_lat", "geocoded_lon", "coord_status"])
print(f"geocoding: {needs_geocoding.sum()} rows flagged, {confirmed_mask.sum()} corrected from checkpoint")

CA_LAT_BOUNDS, CA_LON_BOUNDS = (32.0, 42.5), (-125.0, -113.5)
before = len(housing_after_type_parsing)
bad_coord_mask = ~(
    housing_after_type_parsing["Latitude"].between(*CA_LAT_BOUNDS)
    & housing_after_type_parsing["Longitude"].between(*CA_LON_BOUNDS)
)
print(f"rows outside CA bounds: {bad_coord_mask.sum()} / {before}")
housing_after_type_parsing = housing_after_type_parsing[~bad_coord_mask]
housing_after_type_parsing = housing_after_type_parsing.drop(columns=[c for c in ["ListingKey", "UnparsedAddress"] if c in housing_after_type_parsing.columns])
print(f"-> {len(housing_after_type_parsing)} rows")

geocoding: 2649 rows flagged, 816 corrected from checkpoint
rows outside CA bounds: 151 / 319870
-> 319719 rows


In [9]:
before = len(housing_after_type_parsing)
housing_after_type_parsing["PropertyAgeYears"] = housing_after_type_parsing["SaleYearMonth"].apply(lambda p: p.year) - housing_after_type_parsing["YearBuilt"]
baths_positive_mask = (housing_after_type_parsing["BathroomsTotalInteger"] > 0).fillna(False).to_numpy()
housing_after_type_parsing["BedBathRatio"] = np.where(
    baths_positive_mask,
    housing_after_type_parsing["BedroomsTotal"].astype("float") / housing_after_type_parsing["BathroomsTotalInteger"].astype("float"),
    np.nan,
)
negative_age_mask = housing_after_type_parsing["PropertyAgeYears"] < 0
housing_after_type_parsing = housing_after_type_parsing[~negative_age_mask]
print(f"engineered PropertyAgeYears/BedBathRatio, dropped {negative_age_mask.sum()} negative-age rows ({before} -> {len(housing_after_type_parsing)})")

engineered PropertyAgeYears/BedBathRatio, dropped 26 negative-age rows (319719 -> 319693)


In [10]:
districts_gdf = gpd.read_file("DistrictAreas2425/DistrictAreas2425.shp")
DISTRICT_NAME_COLUMN = "DistrictNa"
districts_for_join = districts_gdf[districts_gdf["DistrictTy"].isin(["Unified", "High"])].copy()

housing_after_type_parsing = housing_after_type_parsing.reset_index(drop=True)
housing_after_type_parsing["_row_id"] = housing_after_type_parsing.index

points_gdf = gpd.GeoDataFrame(
    housing_after_type_parsing,
    geometry=gpd.points_from_xy(housing_after_type_parsing["Longitude"], housing_after_type_parsing["Latitude"]),
    crs="EPSG:4326",
)
if points_gdf.crs != districts_for_join.crs:
    points_gdf = points_gdf.to_crs(districts_for_join.crs)

joined = gpd.sjoin(points_gdf, districts_for_join[[DISTRICT_NAME_COLUMN, "DistrictTy", "geometry"]], how="left", predicate="within")
joined = joined.drop(columns=["geometry", "index_right"])

n_rows_before_dedup = len(joined)
type_rank = {"Unified": 0, "High": 1}
joined["_type_rank"] = joined["DistrictTy"].map(type_rank).fillna(2)
joined = joined.sort_values(["_row_id", "_type_rank"]).drop_duplicates(subset=["_row_id"], keep="first")
assert len(joined) == len(housing_after_type_parsing), "row count changed after join+de-dup"
print(f"school district join: {n_rows_before_dedup} rows before de-dup, {n_rows_before_dedup - len(joined)} duplicates removed -> {len(joined)} rows")

joined = joined.drop(columns=["_row_id", "_type_rank", "DistrictTy"])
housing_final = pd.DataFrame(joined).rename(columns={DISTRICT_NAME_COLUMN: "SchoolDistrictJoined"})
housing_final = housing_final.drop(columns=["HighSchoolDistrict"])
print(f"pre-feature-engineering shape: {housing_final.shape}")

school district join: 319693 rows before de-dup, 0 duplicates removed -> 319693 rows
pre-feature-engineering shape: (319693, 27)


## 10. New Feature #1: Cyclical Month Encoding + Time Trend

`SaleYearMonth` exists purely as the split key in wk9 -- it's never in `feature_columns`, so
the model has zero information about *when* within the year a sale happened, or where in the
overall time window it falls. Two additions, both known before any sale outcome so neither
carries leakage risk:

- `SaleMonthSin` / `SaleMonthCos`: the calendar month (1-12) encoded on a circle so December
  and January are adjacent rather than 11 apart, per the best-practices doc's exact guidance.
- `MonthsSinceStart`: a plain integer counting months from the earliest sale in the dataset --
  a continuous trend a model can use to separate a real price effect from broader market
  appreciation/decline over the (now 24-month) training window. This is a direct response to
  the wk9 Reflection's suspicion that Linear Regression's larger-than-expected degradation was
  partly a symptom of having no temporal signal at all to work with over a longer window.

In [11]:
housing_final["SaleMonth"] = housing_final["SaleYearMonth"].apply(lambda p: p.month)
housing_final["SaleMonthSin"] = np.sin(2 * np.pi * housing_final["SaleMonth"] / 12)
housing_final["SaleMonthCos"] = np.cos(2 * np.pi * housing_final["SaleMonth"] / 12)

period_ordinal = housing_final["SaleYearMonth"].apply(lambda p: p.year * 12 + p.month)
housing_final["MonthsSinceStart"] = period_ordinal - period_ordinal.min()

print(housing_final[["SaleYearMonth", "SaleMonth", "SaleMonthSin", "SaleMonthCos", "MonthsSinceStart"]].drop_duplicates().sort_values("MonthsSinceStart").head(3))
print("...")
print(f"MonthsSinceStart range: 0 to {housing_final['MonthsSinceStart'].max()}")

      SaleYearMonth  SaleMonth  SaleMonthSin  SaleMonthCos  MonthsSinceStart
0           2024-01          1      0.500000  8.660254e-01                 0
8306        2024-02          2      0.866025  5.000000e-01                 1
17857       2024-03          3      1.000000  6.123234e-17                 2
...
MonthsSinceStart range: 0 to 28


## 11. New Feature #2: ZIP/Area Price-per-Sqft Comps  ·  Fit on Training Data Only

Per Section 05 (Locational Features): a neighborhood-level $/sqft aggregate is closer to how
an actual comps-based valuation works than an opaque target-encoded category mean. This has
to be fit on training data only and joined forward, exactly like the outlier thresholds in
Section 12 -- the same leakage-prevention pattern, applied to a different feature. It can't
live inside the `TargetEncoder`/`ColumnTransformer` pipeline because it's not a simple
category encoding, it's a derived numeric aggregate keyed on `PostalCode` with a two-level
fallback for sparse ZIPs, so it's computed by hand at the same point in the notebook the
outlier thresholds are, right after the chronological split and before any model sees the
data.

Fallback logic: a ZIP code with fewer than `MIN_ZIP_TRAIN_N` training-set sales gets its
median $/sqft from `MLSAreaMajor` instead (a coarser, more stable geography), and any area
that's *also* sparse falls back to the global training-set median. This is the "group-wise
imputation" pattern Section 06 recommends, applied to a comps feature rather than a raw
column.

In [12]:
MIN_ZIP_TRAIN_N = 15

def attach_price_per_sqft_comps(train_df, *other_dfs):
    train_pps = train_df["ClosePrice"] / train_df["LivingArea"].replace(0, np.nan)
    train_with_pps = train_df.assign(_pps=train_pps)

    zip_counts = train_with_pps.groupby("PostalCode")["_pps"].count()
    valid_zips = zip_counts[zip_counts >= MIN_ZIP_TRAIN_N].index
    zip_comps = train_with_pps[train_with_pps["PostalCode"].isin(valid_zips)].groupby("PostalCode")["_pps"].median()
    zip_comps.name = "ZipMedianPricePerSqft"

    area_comps = train_with_pps.groupby("MLSAreaMajor")["_pps"].median()
    area_comps.name = "AreaMedianPricePerSqft"

    global_median = train_with_pps["_pps"].median()

    def _apply(df):
        out = df.merge(zip_comps, on="PostalCode", how="left")
        out = out.merge(area_comps, on="MLSAreaMajor", how="left")
        out["ZipMedianPricePerSqft"] = out["ZipMedianPricePerSqft"].fillna(out["AreaMedianPricePerSqft"]).fillna(global_median)
        return out.drop(columns=["AreaMedianPricePerSqft"])

    n_zips_total = train_df["PostalCode"].nunique()
    print(f"comps fit on train: {len(valid_zips)}/{n_zips_total} ZIPs have >= {MIN_ZIP_TRAIN_N} training sales "
          f"and get their own median; the rest fall back to their MLSAreaMajor (or the global median "
          f"${global_median:.0f}/sqft if that's sparse too)")
    return tuple(_apply(d) for d in (train_df,) + other_dfs)

## 12. New Feature #3: Distance to Nearest Major CA Employment Center

Per Section 05: distance to major employment centers computed from lat/long. Unlike the comps
feature above, this is a pure geometric fact about a fixed set of coordinates -- it doesn't
depend on `ClosePrice` or any other outcome, so it carries no leakage risk regardless of which
rows land in train, val, or test, and can be computed once on the full dataset rather than
per-split. Ten major CA population/employment centers, haversine distance to the nearest one.

In [13]:
MAJOR_CA_CENTERS = {
    "Los Angeles": (34.0522, -118.2437), "San Francisco": (37.7749, -122.4194),
    "San Diego": (32.7157, -117.1611), "San Jose": (37.3382, -121.8863),
    "Sacramento": (38.5816, -121.4944), "Fresno": (36.7378, -119.7871),
    "Oakland": (37.8044, -122.2712), "Long Beach": (33.7701, -118.1937),
    "Bakersfield": (35.3733, -119.0187), "Anaheim": (33.8366, -117.9143),
}

def haversine_miles(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1r, lon1r, lat2r, lon2r = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2r - lat1r, lon2r - lon1r
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

distances = np.column_stack([
    haversine_miles(housing_final["Latitude"], housing_final["Longitude"], lat, lon)
    for lat, lon in MAJOR_CA_CENTERS.values()
])
housing_final["DistanceToNearestMajorCenter"] = distances.min(axis=1)
nearest_idx = distances.argmin(axis=1)
housing_final["NearestMajorCenter"] = np.array(list(MAJOR_CA_CENTERS.keys()))[nearest_idx]

print(housing_final["DistanceToNearestMajorCenter"].describe())
print()
print(housing_final["NearestMajorCenter"].value_counts())

count    319693.000000
mean         30.098806
std          25.053651
min           0.002785
25%          10.423996
50%          22.236513
75%          43.539763
max         274.001689
Name: DistanceToNearestMajorCenter, dtype: float64

NearestMajorCenter
Anaheim          111823
Los Angeles       65663
San Diego         44298
San Jose          27580
Oakland           27101
Long Beach        13405
Sacramento         8852
Bakersfield        8816
Fresno             7263
San Francisco      4892
Name: count, dtype: int64


## 13. New Feature #4: Missing-Indicator Flags

Per Section 06: missingness itself may be informative, and zero-filling `AssociationFee` /
`GarageSpaces` (wk9 Fix #8) discards that signal entirely -- a $0 HOA fee and a *missing* HOA
fee are being treated identically by the model right now. Adding explicit flags lets the
model use "was this reported at all" as a feature in its own right, on top of the zero-filled
value. This is a transform of columns that already exist, computed before any split, so it
carries no leakage risk (missingness is a fact about the row, not a statistic learned from
other rows).

In [14]:
housing_final["AssociationFeeMissing"] = housing_final["AssociationFee"].isna().astype(int)
housing_final["GarageSpacesMissing"] = housing_final["GarageSpaces"].isna().astype(int)

print(f"AssociationFee missing: {housing_final['AssociationFeeMissing'].mean():.1%} of rows")
print(f"GarageSpaces missing: {housing_final['GarageSpacesMissing'].mean():.1%} of rows")

os.makedirs("CRMLSCleaned", exist_ok=True)
housing_final.to_csv("CRMLSCleaned/housing_m5_pre_split.csv", index=False)
print(f"\nfinal m5 pre-split shape: {housing_final.shape}")
print("saved CRMLSCleaned/housing_m5_pre_split.csv")

AssociationFee missing: 30.2% of rows
GarageSpaces missing: 3.7% of rows

final m5 pre-split shape: (319693, 35)
saved CRMLSCleaned/housing_m5_pre_split.csv


## 14. Split, Comps, and Outlier Filter

Reuses wk9's chronological split function unchanged, and its `N_TRAIN_MONTHS=24` finding for
this data snapshot. Comps (Section 11) and the outlier percentile thresholds (same 0.5/99.5
pattern as wk9) are both fit on the training split only and applied to val/test -- two
instances of the same fit-on-train discipline.

In [15]:
def chronological_train_val_test_split(df, period_col="SaleYearMonth", n_train_months=None):
    periods = sorted(df[period_col].dropna().unique())
    test_period = periods[-1]
    val_period = periods[-2]
    train_periods = periods[:-2]
    if n_train_months is not None:
        train_periods = train_periods[-n_train_months:]
    train_df = df[df[period_col].isin(train_periods)].sort_values(period_col).reset_index(drop=True)
    val_df = df[df[period_col] == val_period].sort_values(period_col).reset_index(drop=True)
    test_df = df[df[period_col] == test_period].sort_values(period_col).reset_index(drop=True)
    return train_df, val_df, test_df

def apply_outlier_thresholds(df, lower, upper, label=None):
    before = len(df)
    filtered = df[(df["ClosePrice"] > lower) & (df["ClosePrice"] < upper)]
    if label:
        print(f"  {label}: {before} -> {len(filtered)} rows")
    return filtered

N_TRAIN_MONTHS = 24  # wk9's finding for this data snapshot; not re-swept here (see Section 1 note)

train_df, val_df, test_df = chronological_train_val_test_split(housing_final, n_train_months=N_TRAIN_MONTHS)
print(f"pre-outlier-filter split: train {train_df.shape}, val {val_df.shape}, test {test_df.shape}")

lower, upper = train_df["ClosePrice"].quantile([0.005, 0.995])
print(f"\noutlier thresholds (fit on m5 training data only): [{lower:,.0f}, {upper:,.0f}]")
train_df = apply_outlier_thresholds(train_df, lower, upper, "train")
val_df = apply_outlier_thresholds(val_df, lower, upper, "val")
test_df = apply_outlier_thresholds(test_df, lower, upper, "test")

train_df, val_df, test_df = attach_price_per_sqft_comps(train_df, val_df, test_df)
train_df = train_df.drop(columns=["PostalCode", "SaleMonth", "NearestMajorCenter"])
val_df = val_df.drop(columns=["PostalCode", "SaleMonth", "NearestMajorCenter"])
test_df = test_df.drop(columns=["PostalCode", "SaleMonth", "NearestMajorCenter"])

print(f"\nfinal m5 split -- train {train_df.shape}, val {val_df.shape}, test {test_df.shape}")

pre-outlier-filter split: train (266220, 35), val (12004, 35), test (12002, 35)

outlier thresholds (fit on m5 training data only): [190,000, 8,100,080]
  train: 266220 -> 263552 rows
  val: 12004 -> 11872 rows
  test: 12002 -> 11876 rows
comps fit on train: 1030/2887 ZIPs have >= 15 training sales and get their own median; the rest fall back to their MLSAreaMajor (or the global median $535/sqft if that's sparse too)

final m5 split -- train (263552, 33), val (11872, 33), test (11876, 33)


## 15. Feature Buckets and Preprocessor (m5)

In [16]:
numeric_median_columns = [
    "Latitude", "Longitude", "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal",
    "LivingArea", "LotSizeSquareFeet", "YearBuilt", "Levels", "Stories",
    "PropertyAgeYears", "BedBathRatio",
    # new in m5:
    "SaleMonthSin", "SaleMonthCos", "MonthsSinceStart",
    "DistanceToNearestMajorCenter", "ZipMedianPricePerSqft",
    "AssociationFeeMissing", "GarageSpacesMissing",
]
numeric_zero_fill_columns = ["AssociationFee", "GarageSpaces"]
categorical_columns = ["City", "CountyOrParish", "MLSAreaMajor", "SchoolDistrictJoined", "Flooring"]
feature_columns = numeric_median_columns + numeric_zero_fill_columns + categorical_columns
non_feature_columns = ["ClosePrice", "SaleYearMonth"]

missing = set(train_df.columns) - set(non_feature_columns) - set(feature_columns)
assert not missing, f"unbucketed columns: {missing}"

def make_preprocessor():
    return ColumnTransformer(transformers=[
        ("numeric_median", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_median_columns),
        ("numeric_zero", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=0)), ("scale", StandardScaler())]), numeric_zero_fill_columns),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("encode", TargetEncoder(random_state=RANDOM_STATE))]), categorical_columns),
    ])

X_train, y_train = train_df[feature_columns], train_df["ClosePrice"]
X_val, y_val = val_df[feature_columns], val_df["ClosePrice"]
X_test, y_test = test_df[feature_columns], test_df["ClosePrice"]
print(f"feature columns: {len(feature_columns)} (7 new vs. wk9's m4: SaleMonthSin, SaleMonthCos, "
      f"MonthsSinceStart, DistanceToNearestMajorCenter, ZipMedianPricePerSqft, AssociationFeeMissing, GarageSpacesMissing)")

feature columns: 31 (7 new vs. wk9's m4: SaleMonthSin, SaleMonthCos, MonthsSinceStart, DistanceToNearestMajorCenter, ZipMedianPricePerSqft, AssociationFeeMissing, GarageSpacesMissing)


## 16. Full Model Lineup on m5

Reuses wk9's winning Random Forest configuration directly (`max_features='sqrt', max_depth=20`)
rather than re-running the ablation -- that question was already answered in wk9 and doesn't
depend on the feature set. XGBoost/LightGBM are re-tuned on the same small grid as wk9.

In [17]:
def evaluate(pipe, X, y):
    preds = pipe.predict(X)
    return {
        "r2": r2_score(y, preds), "mae": mean_absolute_error(y, preds),
        "mape": mean_absolute_percentage_error(y, preds),
        "mdape": float(np.median(np.abs((y - preds) / y))),
    }

models = {}

lr_pipeline = Pipeline([("preprocess", make_preprocessor()), ("model", LinearRegression())])
lr_pipeline.fit(X_train, y_train)
models["LinearRegression"] = lr_pipeline

dt_pipeline = Pipeline([("preprocess", make_preprocessor()), ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))])
dt_pipeline.fit(X_train, y_train)
models["DecisionTree"] = dt_pipeline

t0 = time.time()
rf_pipeline = Pipeline([
    ("preprocess", make_preprocessor()),
    ("model", RandomForestRegressor(n_estimators=200, max_features="sqrt", max_depth=20, random_state=RANDOM_STATE, n_jobs=-1)),
])
rf_pipeline.fit(X_train, y_train)
print(f"RandomForest fit in {time.time()-t0:.1f}s")
models["RandomForest"] = rf_pipeline
os.makedirs("models", exist_ok=True)
joblib.dump(rf_pipeline, "models/rf_m5.pkl")

xgb_param_grid = [
    {"n_estimators": 300, "max_depth": 4, "learning_rate": 0.10},
    {"n_estimators": 500, "max_depth": 5, "learning_rate": 0.05},
    {"n_estimators": 800, "max_depth": 6, "learning_rate": 0.03},
]
lgbm_param_grid = [
    {"n_estimators": 300, "max_depth": -1, "num_leaves": 31, "learning_rate": 0.10},
    {"n_estimators": 500, "max_depth": -1, "num_leaves": 63, "learning_rate": 0.05},
    {"n_estimators": 800, "max_depth": -1, "num_leaves": 127, "learning_rate": 0.03},
]

def tune_on_val(model_class, param_grid, label, **extra):
    best_params, best_val_r2, best_pipeline = None, -np.inf, None
    for params in param_grid:
        pipe = Pipeline([("preprocess", make_preprocessor()), ("model", model_class(random_state=RANDOM_STATE, **params, **extra))])
        pipe.fit(X_train, y_train)
        val_r2 = r2_score(y_val, pipe.predict(X_val))
        print(f"  {label} {params} -> val R2={val_r2:.4f}")
        if val_r2 > best_val_r2:
            best_params, best_val_r2, best_pipeline = params, val_r2, pipe
    print(f"  best {label}: {best_params} (val R2={best_val_r2:.4f})")
    return best_pipeline

xgb_pipeline = tune_on_val(XGBRegressor, xgb_param_grid, "XGBoost")
models["XGBoost"] = xgb_pipeline
joblib.dump(xgb_pipeline, "models/xgb_m5.pkl")

lgbm_pipeline = tune_on_val(LGBMRegressor, lgbm_param_grid, "LightGBM", verbosity=-1)
models["LightGBM"] = lgbm_pipeline
joblib.dump(lgbm_pipeline, "models/lgbm_m5.pkl")

overall_rows = []
predictions = {}
for name, pipe in models.items():
    preds = pipe.predict(X_test)
    predictions[name] = preds
    m = evaluate(pipe, X_test, y_test)
    overall_rows.append({"model": name, "band": "overall", "n_rows": len(y_test), **m})
    print(f"{name:>16s}: R2={m['r2']:.4f}  MAE=${m['mae']:,.0f}  MAPE={m['mape']:.2%}  MdAPE={m['mdape']:.2%}")

overall_df = pd.DataFrame(overall_rows)

C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pa

RandomForest fit in 6.7s


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


  XGBoost {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1} -> val R2=0.8894


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


...

  LightGBM {'n_estimators': 300, 'max_depth': -1, 'num_leaves': 31, 'learning_rate': 0.1} -> val R2=0.9013


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


  LightGBM {'n_estimators': 500, 'max_depth': -1, 'num_leaves': 63, 'learning_rate': 0.05} -> val R2=0.9055


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


  LightGBM {'n_estimators': 800, 'max_depth': -1, 'num_leaves': 127, 'learning_rate': 0.03} -> val R2=0.9084
  best LightGBM: {'n_estimators': 800, 'max_depth': -1, 'num_leaves': 127, 'learning_rate': 0.03} (val R2=0.9084)
LinearRegression: R2=0.7181  MAE=$322,394  MAPE=29.96%  MdAPE=22.66%
    DecisionTree: R2=0.7871  MAE=$224,990  MAPE=16.43%  MdAPE=10.79%
    RandomForest: R2=0.8967  MAE=$156,828  MAPE=11.53%  MdAPE=7.57%
         XGBoost: R2=0.9023  MAE=$159,221  MAPE=11.96%  MdAPE=8.34%
        LightGBM: R2=0.9094  MAE=$152,550  MAPE=11.55%  MdAPE=7.97%


## 17. Headline Comparison: m4 (wk9) vs. m5 (Domain Features)

The m4 numbers are the corrected wk9 results (this machine, this data snapshot -- see wk9's
Reflection for the data-snapshot story), hardcoded here for a direct read.

In [18]:
m4_baseline = {
    "LinearRegression": {"r2": 0.718250, "mae": 320433.30, "mape": 0.296031, "mdape": 0.2219},
    "DecisionTree":     {"r2": 0.797306, "mae": 218923.14, "mape": 0.161338, "mdape": 0.1065},
    "RandomForest":     {"r2": 0.889592, "mae": 163075.33, "mape": 0.119272, "mdape": 0.0794},
    "XGBoost":          {"r2": 0.895772, "mae": 166401.91, "mape": 0.124986, "mdape": 0.0868},
    "LightGBM":         {"r2": 0.905303, "mae": 156417.52, "mape": 0.117759, "mdape": 0.0815},
}

comparison_rows = []
for name in models:
    m4 = m4_baseline[name]
    m5 = overall_df.set_index("model").loc[name]
    comparison_rows.append({
        "model": name,
        "m4_r2": m4["r2"], "m5_r2": m5["r2"], "r2_delta": m5["r2"] - m4["r2"],
        "m4_mape": m4["mape"], "m5_mape": m5["mape"], "mape_delta_pp": (m5["mape"] - m4["mape"]) * 100,
        "m4_mae": m4["mae"], "m5_mae": m5["mae"], "mae_delta": m5["mae"] - m4["mae"],
    })
comparison_df = pd.DataFrame(comparison_rows).set_index("model")
comparison_df = comparison_df.sort_values("m5_r2", ascending=False)
comparison_df

,m4_r2,m5_r2,r2_delta,m4_mape,m5_mape,mape_delta_pp,m4_mae,m5_mae,mae_delta
model,,,,,,,,,
LightGBM,0.905303,0.909440,0.004137,0.117759,0.115474,-0.228511,156417.52,152550.385914,-3867.134086
XGBoost,0.895772,0.902324,0.006552,0.124986,0.119581,-0.540507,166401.91,159220.650051,-7181.259949
RandomForest,0.889592,0.896701,0.007109,0.119272,0.115313,-0.395925,163075.33,156828.067381,-6247.262619
DecisionTree,0.797306,0.787068,-0.010238,0.161338,0.164292,0.295421,218923.14,224990.030700,6066.890700
LinearRegression,0.718250,0.718124,-0.000126,0.296031,0.299625,0.359372,320433.30,322394.480210,1961.180210


## 18. Do the New Features Actually Matter? Feature Importance on m5

In [19]:
best_model_name = overall_df.set_index("model")["r2"].idxmax()
best_pipeline = models[best_model_name]
print(f"best m5 model: {best_model_name}")

fitted_preprocessor = best_pipeline.named_steps["preprocess"]
raw_importances = best_pipeline.named_steps["model"].feature_importances_

expanded_names = []
expanded_names += numeric_median_columns
expanded_names += numeric_zero_fill_columns
expanded_names += categorical_columns  # TargetEncoder: one output column per input column

importance_df = pd.DataFrame({"feature": expanded_names, "importance": raw_importances})
importance_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)
importance_df["rank"] = importance_df.index + 1

NEW_FEATURES = {"SaleMonthSin", "SaleMonthCos", "MonthsSinceStart", "DistanceToNearestMajorCenter",
                 "ZipMedianPricePerSqft", "AssociationFeeMissing", "GarageSpacesMissing"}
importance_df["is_new_m5_feature"] = importance_df["feature"].isin(NEW_FEATURES)

importance_df.to_csv("Deliverables/wk10_feature_importance.csv", index=False)
print(importance_df.to_string(index=False))

best m5 model: LightGBM
                     feature  importance  rank  is_new_m5_feature
                  LivingArea        9977     1              False
           LotSizeSquareFeet        9531     2              False
DistanceToNearestMajorCenter        8201     3               True
       ZipMedianPricePerSqft        7268     4               True
                    Latitude        6203     5              False
                   Longitude        6107     6              False
                        City        5494     7              False
                MLSAreaMajor        5413     8              False
                   YearBuilt        5270     9              False
                    Flooring        4355    10              False
        SchoolDistrictJoined        4144    11              False
              AssociationFee        3446    12              False
            PropertyAgeYears        3318    13              False
            MonthsSinceStart        2709    14      

If the new features land near the top, they're pulling real weight, not just adding noise the
model has to sift through; if they land near the bottom, that's useful to know too -- it means
the signal they carry was already mostly available through some other feature (e.g.
`ZipMedianPricePerSqft` overlapping with what `MLSAreaMajor`'s target encoding already
captures), and it's a legitimate, evidence-based reason to consider dropping them rather than
carrying dead weight. Read the printed ranking above rather than assuming; that's exactly the
kind of unverified assumption this whole notebook series has been trying to stop making.

## 19. Rolling-Origin Backtest

Per Section 01: a single train/test cutoff can be lucky or unlucky. Every number in wk9 and in
this notebook so far comes from one cutoff (test = the most recent complete month). This
section re-runs the winning model at two additional, earlier cutoffs -- pretending the dataset
ends one and two months earlier than it actually does -- and checks whether R2/MAPE/MdAPE stay
in a similar range.

Scope note: this reuses the best model's already-chosen hyperparameters rather than re-tuning
at each cutoff (re-tuning three times would triple the cost for a check that's about pipeline
*stability*, not further optimization). Also reuses `N_TRAIN_MONTHS=24` at every cutoff for
the same reason.

In [20]:
def run_backtest_cutoff(full_df, months_to_hold_back, model_class, best_params, extra_kwargs):
    periods = sorted(full_df["SaleYearMonth"].dropna().unique())
    cutoff_df = full_df[full_df["SaleYearMonth"].isin(periods[: len(periods) - months_to_hold_back])] if months_to_hold_back > 0 else full_df

    tr, va, te = chronological_train_val_test_split(cutoff_df, n_train_months=N_TRAIN_MONTHS)
    lo, hi = tr["ClosePrice"].quantile([0.005, 0.995])
    tr = apply_outlier_thresholds(tr, lo, hi)
    va = apply_outlier_thresholds(va, lo, hi)
    te = apply_outlier_thresholds(te, lo, hi)
    tr, va, te = attach_price_per_sqft_comps(tr, va, te)
    for d in (tr, va, te):
        d.drop(columns=["PostalCode", "SaleMonth", "NearestMajorCenter"], inplace=True)

    pipe = Pipeline([("preprocess", make_preprocessor()), ("model", model_class(random_state=RANDOM_STATE, **best_params, **extra_kwargs))])
    pipe.fit(tr[feature_columns], tr["ClosePrice"])
    m = evaluate(pipe, te[feature_columns], te["ClosePrice"])
    test_period = sorted(te["SaleYearMonth"].unique())[0]
    return {"test_period": str(test_period), "months_held_back": months_to_hold_back, "n_test_rows": len(te), **m}

if best_model_name == "LightGBM":
    backtest_model_class, backtest_params, backtest_extra = LGBMRegressor, lgbm_pipeline.named_steps["model"].get_params(), {"verbosity": -1}
    backtest_params = {k: v for k, v in backtest_params.items() if k in {"n_estimators", "max_depth", "num_leaves", "learning_rate"}}
elif best_model_name == "XGBoost":
    backtest_model_class, backtest_params, backtest_extra = XGBRegressor, xgb_pipeline.named_steps["model"].get_params(), {}
    backtest_params = {k: v for k, v in backtest_params.items() if k in {"n_estimators", "max_depth", "learning_rate"}}
else:
    backtest_model_class, backtest_params, backtest_extra = RandomForestRegressor, {"n_estimators": 200, "max_features": "sqrt", "max_depth": 20}, {"n_jobs": -1}

backtest_rows = [{"test_period": str(sorted(test_df["SaleYearMonth"].unique())[0]), "months_held_back": 0, "n_test_rows": len(test_df), **overall_df.set_index("model").loc[best_model_name][["r2", "mae", "mape", "mdape"]].to_dict()}]
for k in [1, 2]:
    row = run_backtest_cutoff(housing_final, k, backtest_model_class, backtest_params, backtest_extra)
    backtest_rows.append(row)
    print(f"cutoff -{k} month(s) (test={row['test_period']}, n={row['n_test_rows']}): "
          f"R2={row['r2']:.4f}  MAPE={row['mape']:.2%}  MdAPE={row['mdape']:.2%}")

backtest_df = pd.DataFrame(backtest_rows)
backtest_df.to_csv("Deliverables/wk10_rolling_origin_backtest.csv", index=False)
backtest_df

comps fit on train: 1029/2893 ZIPs have >= 15 training sales and get their own median; the rest fall back to their MLSAreaMajor (or the global median $535/sqft if that's sparse too)


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


cutoff -1 month(s) (test=2026-04, n=11870): R2=0.9078  MAPE=11.75%  MdAPE=8.01%
comps fit on train: 1028/2901 ZIPs have >= 15 training sales and get their own median; the rest fall back to their MLSAreaMajor (or the global median $535/sqft if that's sparse too)


C:\Users\kikoh\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


cutoff -2 month(s) (test=2026-03, n=11015): R2=0.9111  MAPE=11.51%  MdAPE=8.13%


,test_period,months_held_back,n_test_rows,r2,mae,mape,mdape
0,2026-05,0,11876,0.909440,152550.385914,0.115474,0.079711
1,2026-04,1,11870,0.907828,152771.977120,0.117468,0.080146
2,2026-03,2,11015,0.911141,145026.595884,0.115069,0.081284


Read the R2/MAPE column-to-column, not just the single "current" row: if all three cutoffs
land in a similar range, that's real evidence this model generalizes across market moments
rather than being tuned to one lucky month, which is the entire point of this check per the
best-practices doc. If one cutoff is a clear outlier, that's worth flagging rather than
picking whichever cutoff looks best and reporting only that one.

## 20. Evaluation on the Team's Prescribed Price Bands

In [21]:
TEAM_PRICE_BAND_EDGES = [0, 500_000, 1_000_000, 2_000_000, np.inf]
TEAM_PRICE_BAND_LABELS = ["under $500K", "$500K-$1M", "$1M-$2M", "$2M+"]
team_bands = pd.cut(y_test, bins=TEAM_PRICE_BAND_EDGES, labels=TEAM_PRICE_BAND_LABELS)

band_rows = []
for name, preds in predictions.items():
    for label in TEAM_PRICE_BAND_LABELS:
        mask = (team_bands == label).values
        if mask.sum() == 0:
            continue
        y_band, preds_band = y_test[mask], preds[mask]
        band_rows.append({
            "model": name, "band": label, "n_rows": int(mask.sum()),
            "r2": np.nan, "mae": mean_absolute_error(y_band, preds_band),
            "mape": mean_absolute_percentage_error(y_band, preds_band),
            "mdape": float(np.median(np.abs((y_band - preds_band) / y_band))),
        })
band_df = pd.DataFrame(band_rows)
print("MAPE by model x band:")
print(band_df.pivot(index="model", columns="band", values="mape")[TEAM_PRICE_BAND_LABELS].round(4))

wk10_metrics_summary = pd.concat([overall_df, band_df], ignore_index=True)[["model", "band", "n_rows", "r2", "mae", "mape", "mdape"]]
wk10_metrics_summary.to_csv("Deliverables/wk10_m5_metrics_summary.csv", index=False)
print(f"\nsaved Deliverables/wk10_m5_metrics_summary.csv ({len(wk10_metrics_summary)} rows)")

MAPE by model x band:
band              under $500K  $500K-$1M  $1M-$2M    $2M+
model                                                    
DecisionTree           0.1806     0.1344   0.1759  0.2117
LightGBM               0.1473     0.0958   0.1166  0.1402
LinearRegression       0.5398     0.3097   0.2122  0.2291
RandomForest           0.1432     0.0939   0.1173  0.1478
XGBoost                0.1487     0.0990   0.1215  0.1480

saved Deliverables/wk10_m5_metrics_summary.csv (25 rows)


## Reflection

**Does the headline gap actually matter?** Section 1 named the biggest structural gap as "no
temporal feature at all." Sections 17-18 are where that claim gets checked against real
numbers rather than left as a plausible-sounding assumption -- read those cells' output before
trusting this summary, since the point of this whole notebook series has been not doing that.

**What the new features are, in one line each, and why each is leakage-safe:** cyclical
month + time trend (known before any sale, zero leakage), ZIP/area price-per-sqft comps
(fit on training data only, same discipline as the outlier thresholds), distance to nearest
major employment center (a fixed geometric fact, not a data-derived statistic -- computable
even for an off-market property with no sale history at all), missing-indicator flags (a
transform of existing columns, not a new external signal).

**The rolling-origin backtest is new methodology, not just a new feature, and it matters for a
different reason.** Every number in wk9, and every number in this notebook up through Section
18, comes from a single train/val/test cutoff. Section 19 is the first time this pipeline has
checked whether its own headline numbers are stable across cutoffs rather than a product of
one lucky (or unlucky) month. That distinction -- a good score on one split vs. a model that
actually generalizes -- is the central argument of the best-practices doc's Section 01, and it
was the one practice from that document not addressed at all until this notebook.

**What I'd flag as still open, honestly:** `N_TRAIN_MONTHS` was reused at 24 rather than
re-swept with the new feature set -- plausible that comps/temporal features shift the
accuracy/window-length tradeoff, not verified here. The comps feature's `MIN_ZIP_TRAIN_N=15`
threshold and the choice of exactly 10 employment centers are both reasonable but somewhat
arbitrary judgment calls, not something separately validated by an ablation the way wk9's RF
`max_features` was. Both are legitimate targets for a future pass rather than settled
questions.